# Spotify User Behavior Analysis

## Objective
This project analyzes Spotify listening history to understand user behavior, engagement patterns, and music consumption trends.

## Scope
The analysis focuses on:
- listening volume over time,
- playback behavior (skip and completion),
- platform usage,
- concentration of listening time across artists.

## Dashboard Connection
The cleaned dataset generated in this notebook is designed to feed the Power BI dashboard included in this project.


## Project Setup

This notebook uses relative paths so the project can be shared through GitHub without depending on a local machine setup.

project structure:

- `data/raw/` → raw Spotify JSON files  
- `data/processed/` → cleaned CSV outputs  
- `dashboard/` → Power BI dashboard  
- `assets/` → dashboard preview image


## Project Setup
This section imports the required libraries and defines the project paths used throughout the workflow.

In [1]:
import json
import os
import pandas as pd
from pathlib import Path

# Project base path (the notebook is stored inside /notebooks)
project_root = Path.cwd().parent

# Path to the folder containing the source JSON files
json_folder_path = project_root / "data" / "raw"

# Path where the combined CSV file will be saved
output_csv_path = project_root / "data" / "processed" / "spotify_listening_history_raw.csv"

# Create the output folder if it does not exist
output_csv_path.parent.mkdir(parents=True, exist_ok=True)

# Assumed encoding (common for JSON files)
encoding = "utf-8"

print("Project root:", project_root)
print("JSON folder:", json_folder_path)
print("Output CSV path:", output_csv_path)


Project root: c:\Users\pablo\OneDrive\Desktop\Projects\Git_Hub_Projects\spotify-user-behavior
JSON folder: c:\Users\pablo\OneDrive\Desktop\Projects\Git_Hub_Projects\spotify-user-behavior\data\raw
Output CSV path: c:\Users\pablo\OneDrive\Desktop\Projects\Git_Hub_Projects\spotify-user-behavior\data\processed\spotify_listening_history_raw.csv


## Data Consolidation
This step combines multiple Spotify JSON files into a single raw CSV file for further analysis and dashboard development.

In [2]:
# List to store combined records
combined_data = []

# Read and combine all JSON files in the folder
for file_name in os.listdir(json_folder_path):
    if file_name.endswith(".json"):  # Check if the file is a JSON file
        file_path = os.path.join(json_folder_path, file_name)

        try:
            # Open the file using the specified encoding
            with open(file_path, "r", encoding=encoding) as f:
                data = json.load(f)  # Load JSON data

                # If the file contains a list of objects
                if isinstance(data, list):
                    combined_data.extend(data)

                # If the file contains a single JSON object
                elif isinstance(data, dict):
                    combined_data.append(data)

        except json.JSONDecodeError as e:
            print(f"Error decoding file {file_name}: {e}")

        except Exception as e:
            print(f"Error reading file {file_name}: {e}")

# Convert combined data into a pandas DataFrame
df = pd.DataFrame(combined_data)

# Export to CSV using UTF-8 with BOM for compatibility (Excel-friendly)
df.to_csv(
    output_csv_path,
    index=False,
    encoding="utf-8-sig",
    sep=";",
    quoting=1
)

print(f"Combined data successfully exported to: '{output_csv_path}'")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

Combined data successfully exported to: 'c:\Users\pablo\OneDrive\Desktop\Projects\Git_Hub_Projects\spotify-user-behavior\data\processed\spotify_listening_history_raw.csv'
Total rows: 278178
Total columns: 19


### Initial Observation
At this stage, the dataset is still raw. The next step is to inspect its structure and identify the fields required for analysis and dashboarding.


#### Load Combined Dataset
The consolidated CSV file is loaded to begin the exploratory analysis phase.

In [3]:
# Load the combined CSV file
df = pd.read_csv(output_csv_path, encoding="utf-8-sig", sep=";")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (278178, 19)


C:\Users\pablo\AppData\Local\Temp\ipykernel_18912\36973185.py:2: DtypeWarning: Columns (9,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(output_csv_path, encoding="utf-8-sig", sep=";")


#### Dataset Structure
In this section, we inspect the dataset columns and dimensions to better understand its structure before starting the analysis.

In [4]:
# Get basic dataset structure information
column_names = list(df.columns)
num_rows, num_columns = df.shape

print("Dataset Overview")
print("-" * 40)
print("Number of columns:", num_columns)
print("Number of rows:", num_rows)
df.head(5)

Dataset Overview
----------------------------------------
Number of columns: 19
Number of rows: 278178


,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,spotify_episode_uri,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
0,2015-05-10T21:39:50Z,Windows 7 (6.1.7601; x86; SP1; S),418334,CO,186.154.56.185,Vuela,Sin Censura,Coctel Veneno,spotify:track:1PufKwsecVzsM4Crx8f9cb,NaN,NaN,NaN,clickrow,trackdone,False,False,False,NaN,False
1,2015-05-10T21:41:17Z,Windows 7 (6.1.7601; x86; SP1; S),87678,CO,186.154.56.185,Game Over,Sin Censura,Coctel Veneno,spotify:track:0vOEu6IW4IAobugPcBmezc,NaN,NaN,NaN,trackdone,clickrow,False,True,False,NaN,False
2,2015-05-10T23:59:03Z,Windows 7 (6.1.7601; x86; SP1; S),44013,CO,186.154.56.185,Vuela,Sin Censura,Coctel Veneno,spotify:track:1PufKwsecVzsM4Crx8f9cb,NaN,NaN,NaN,clickrow,backbtn,False,True,False,NaN,False
3,2015-05-10T23:59:09Z,Windows 7 (6.1.7601; x86; SP1; S),2832,CO,186.154.56.185,Laura,Sin Censura,Coctel Veneno,spotify:track:4OSulsWL5eh9WMgwUKaFwB,NaN,NaN,NaN,backbtn,fwdbtn,False,True,False,NaN,False
4,2015-05-11T01:32:18Z,Windows 7 (6.1.7601; x86; SP1; S),506618,CO,186.154.56.185,Vuela,Sin Censura,Coctel Veneno,spotify:track:1PufKwsecVzsM4Crx8f9cb,NaN,NaN,NaN,fwdbtn,trackdone,False,False,False,NaN,False


#### Playback Behavior Analysis
This section examines the most common playback starting points and session ending behaviors to better understand how listening activity begins and ends.

In [5]:
# Count the frequency of values in playback behavior fields
reason_start_counts = df["reason_start"].value_counts()
reason_end_counts = df["reason_end"].value_counts()

print("Reason Start Counts")
print(reason_start_counts)

print("\nReason End Counts")
print(reason_end_counts)

Reason Start Counts
reason_start
trackdone     182042
fwdbtn         48260
clickrow       23538
playbtn        12133
backbtn         5041
appload         3591
remote          2941
trackerror       553
unknown           79
Name: count, dtype: int64

Reason End Counts
reason_end
trackdone                       186967
fwdbtn                           48188
endplay                          27193
unexpected-exit-while-paused      5274
backbtn                           4967
logout                            2456
remote                            2233
unexpected-exit                    566
unknown                            185
trackerror                          87
clickrow                            62
Name: count, dtype: int64


**Initial Interpretation:**  
The `reason_start` and `reason_end` variables provide useful context about playback behavior. They can help identify whether listening sessions are mostly user-initiated, algorithm-driven, or completed naturally, which supports the interpretation of engagement metrics in the dashboard.

#### Listening Time Calculation
This section calculates the total listening time using the `ms_played` variable and converts it into more interpretable formats such as minutes and hours.

In [6]:
# Calculate total listening time from the "ms_played" column
total_ms_played = df["ms_played"].sum()

# Convert total listening time into minutes and hours
total_minutes_played = total_ms_played / 60000
total_hours_played = total_ms_played / 3600000

print("Listening Time Summary")
print("-" * 30)
print(f"Total milliseconds played: {total_ms_played:,.0f}")
print(f"Total minutes played: {total_minutes_played:,.2f}")
print(f"Total hours played: {total_hours_played:,.2f}")

Listening Time Summary
------------------------------
Total milliseconds played: 47,132,874,546
Total minutes played: 785,547.91
Total hours played: 13,092.47


**Initial Interpretation:**  
Total listening time provides a high-level measure of overall consumption. Converting this metric into minutes and hours makes it easier to interpret and connects directly to one of the main KPIs in the Power BI dashboard.

#### Platform Usage Analysis
This section identifies the platforms used for listening activity. Understanding platform distribution helps reveal the dominant listening channel and supports the interpretation of user behavior patterns in the dashboard.

In [7]:
# Count the frequency of each platform
platform_counts = df["platform"].value_counts()

print("Platform Counts")
print(platform_counts)

Platform Counts
platform
ios                                                 53640
Android OS 9 API 28 (HUAWEI, SNE-LX3)               39475
Android OS 10 API 29 (HUAWEI, SNE-LX3)              29412
Windows 8.1 (6.3.9600; x64)                         20597
Android OS 8.1.0 API 27 (HUAWEI, SNE-LX3)           19507
                                                    ...  
iOS 15.2 (iPhone12,8)                                  33
iOS 0.4.0 (unknown) [arm 0]                            32
Android OS 9 API 28 (samsung, SM-A305G)                32
web_player windows 10;chrome 71.0;desktopinstall       15
Windows 10 (10.0.17134; x64)                            2
Name: count, Length: 62, dtype: int64


**Initial Interpretation:**  
Platform usage helps identify the main listening channel and provides context for understanding how consumption takes place. This variable is later used in the Power BI dashboard to highlight the dominant platform and support behavioral insights.

#### Data Preparation for Power BI
This section prepares the final dataset used in the Power BI dashboard. It includes timestamp splitting, feature creation, column cleanup, content classification, and platform grouping to make the data easier to analyze and visualize.

In [8]:
# Paths for the Power BI input dataset
input_csv_path = project_root / "data" / "processed" / "spotify_listening_history_raw.csv"
output_csv_path = project_root / "data" / "processed" / "spotify_list_clean.csv"

# Load the original combined CSV file
df = pd.read_csv(input_csv_path, sep=";", encoding="utf-8-sig", on_bad_lines="skip")

# Split the "ts" column into two new columns: "fecha" and "hora"
df[["fecha", "hora"]] = df["ts"].str.split("T", expand=True)
df["hora"] = df["hora"].str.replace("Z", "", regex=False)

# Create a new metric from "ms_played" in minutes
df["minutos_played"] = df["ms_played"] / 60000

# Keep the original formatting logic for Power BI compatibility
df["minutos_played"] = df["minutos_played"].map("{:.2f}".format).str.replace(".", ",")

# Remove unnecessary columns
columnas_a_eliminar = [
    "conn_country", "ip_addr", "spotify_track_uri", "spotify_episode_uri",
    "offline", "offline_timestamp", "incognito_mode"
]
df = df.drop(columns=columnas_a_eliminar, errors="ignore")

# Create the "tipo_contenido" column
df["tipo_contenido"] = df["master_metadata_track_name"].apply(
    lambda x: "Music" if pd.notnull(x) and str(x).strip() != "" else "Podcast"
)

# Create the "Type_platform" column based on "platform"
def get_platform_type(platform):
    if pd.isna(platform):
        return "Other"
    
    platform = platform.lower()

    if "android" in platform:
        return "Phone"
    elif "ios" in platform:
        return "Phone"
    elif "web_player" in platform:
        return "Web"
    elif "partner windows_tv" in platform or "xbox" in platform:
        return "Console"
    elif "windows" in platform or "windows 7" in platform or "windows 8" in platform or "windows 10" in platform:
        return "Pc"
    elif "tizen" in platform:
        return "Tv"
    elif "amazon_salmon" in platform:
        return "Alexa"
    else:
        return "Other"

# Apply the function to create "Type_platform"
df["Type_platform"] = df["platform"].apply(get_platform_type)

# Reorder columns for the final Power BI dataset
columnas_finales = [
    "fecha", "hora", "platform", "ms_played", "minutos_played",
    "master_metadata_track_name", "master_metadata_album_artist_name",
    "master_metadata_album_album_name", "episode_name", "episode_show_name",
    "reason_start", "reason_end", "shuffle", "skipped", "tipo_contenido", "Type_platform"
]
df = df[columnas_finales]

# Export the final dataset for Power BI
df.to_csv(output_csv_path, index=False, sep=";", encoding="utf-8-sig", decimal=",")

print(f"Power BI dataset saved to: {output_csv_path}")
print("Final dataset shape:", df.shape)
print("This file is used as the input source for the Power BI dashboard.")


C:\Users\pablo\AppData\Local\Temp\ipykernel_18912\1120389759.py:6: DtypeWarning: Columns (9,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_csv_path, sep=";", encoding="utf-8-sig", on_bad_lines="skip")


Power BI dataset saved to: c:\Users\pablo\OneDrive\Desktop\Projects\Git_Hub_Projects\spotify-user-behavior\data\processed\spotify_list_clean.csv
Final dataset shape: (278178, 16)
This file is used as the input source for the Power BI dashboard.


#### Final Dataset for Visualization
The dataset has been cleaned, structured, and enriched with additional variables such as content type, platform grouping, and listening time metrics.

This final version is optimized for analysis and serves as the input for the Power BI dashboard.

### Power BI Connection Notes

The cleaned dataset generated in Python is used as the input source for the Power BI dashboard.

#### Output file generated by this notebook
The final file is exported to:

`data/processed/spotify_list_clean.csv`

#### How to connect the file in Power BI
To use this dataset in Power BI:

1. Open the `.pbix` file from the `dashboard` folder  
2. Go to **Transform Data**  
3. Open the query that loads the CSV file  
4. Update the file path if the project folder has been moved to a different local directory  
5. Refresh the model to load the latest cleaned dataset  

#### Important note
Power BI Desktop uses local file paths when importing CSV files.  
For that reason, if the project is moved to another computer or folder, the source path in Power BI may need to be updated manually.

#### Power BI Integration

The dashboard enables:
- Visualization of listening trends over time  
- Analysis of platform usage distribution  
- Monitoring of engagement metrics such as skip rate and completion rate  
- Exploration of artist-level consumption patterns  

This integration allows raw listening data to be transformed into actionable insights through visual analytics.

#### Key Insights

Based on the analysis and dashboard results, the following behavioral patterns were identified:

- **Listening peaks during Q4 (October–December)**, suggesting a seasonal pattern in music consumption  
- **Mobile devices dominate usage (~80%+)**, indicating strong on-the-go listening behavior  
- **Low skip rate (~6%)** suggests users are highly engaged with the content they choose  
- **High completion rate (~67%)** confirms consistent playback behavior  
- **Listening is concentrated among a small group of artists**, indicating preference-driven consumption  

These insights highlight how users interact with music in terms of time, platform, and engagement level.

#### Conclusion

This project demonstrates how raw behavioral data can be transformed into a structured analytical dataset and meaningful insights.

Through data cleaning, feature engineering, and metric definition, the analysis provides a clear view of listening behavior and engagement patterns.

The combination of Python for data preparation and Power BI for visualization enables a complete analytical workflow, from raw data to interactive insights.